# Pneumonia Detection — Classical ML Pipeline (joblib)

**Approach**: Scikit-learn models on handcrafted image features.  
**Output**: `best_model_pipeline.joblib` — a single portable file containing the full preprocessing + classification pipeline.  
**Dataset**: RSNA Pneumonia Detection Challenge (shared `data/` directory with `pneumonia_detection.ipynb`).

### Why joblib?
- Scikit-learn pipelines are self-contained: scaler + model in one `.joblib` file.
- `joblib.load()` requires only scikit-learn — no TensorFlow, no GPU.
- Deployable to any Python environment (REST API, CLI, Lambda, Docker, etc.).
- Provides a classical ML baseline to compare against the CNN (AUC 0.768).

### Pipeline
```
DICOM → resize 64×64 → flatten (4096) + 5 stats → StandardScaler → Classifier → P(pneumonia)
```

In [ ]:
# Install all required packages.
# Run once; skip on subsequent sessions.
%pip install pydicom opencv-python==4.9.0.80 "numpy>=1.26,<2.0" pandas scikit-learn joblib matplotlib seaborn "scipy<1.14" tqdm --quiet

---
## Section 1: Imports & Configuration

In [ ]:
# Core
import os
import warnings
import numpy as np
import pandas as pd
import pydicom
import cv2
import joblib
from tqdm import tqdm

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn — pipeline, models, metrics
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore", category=UserWarning)
np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────────────────────
# Shares the same data/ directory as pneumonia_detection.ipynb in this folder.
DATA_DIR    = "data"
IMG_DIR     = os.path.join(DATA_DIR, "train_images")   # extracted DICOMs (~26K .dcm files)
LABELS_CSV  = os.path.join(DATA_DIR, "stage_2_train_labels.csv")
CLASS_CSV   = os.path.join(DATA_DIR, "stage_2_detailed_class_info.csv")

# ── Feature config ────────────────────────────────────────────────────────────
FEATURE_IMG_SIZE = (64, 64)   # 64×64 = 4096 pixel features per image
N_FEATURES       = FEATURE_IMG_SIZE[0] * FEATURE_IMG_SIZE[1] + 5  # +5 global stats
FEATURE_CACHE    = "features_cache.npz"  # avoids re-extraction on repeat runs (~110 MB)

# ── Output ────────────────────────────────────────────────────────────────────
MODEL_OUTPUT = "best_model_pipeline.joblib"

print(f"Data dir exists : {os.path.exists(DATA_DIR)}")
print(f"IMG_DIR exists  : {os.path.exists(IMG_DIR)}")
print(f"Feature size    : {N_FEATURES} values per image")

---
## Section 2: Data Loading

In [ ]:
# train_labels has one row per bounding box — pneumonia patients appear multiple times.
# Deduplicate to one row per patient before splitting to prevent data leakage.
train_labels = pd.read_csv(LABELS_CSV)
class_info   = pd.read_csv(CLASS_CSV)

patient_df = (
    train_labels[["patientId", "Target"]]
    .drop_duplicates(subset="patientId")   # one row per patient
    .merge(class_info.drop_duplicates(subset="patientId"), on="patientId", how="left")
    .reset_index(drop=True)
)

# Only keep patients whose DICOM file is actually on disk
on_disk = set(os.path.splitext(f)[0] for f in os.listdir(IMG_DIR) if f.endswith(".dcm"))
patient_df = patient_df[patient_df["patientId"].isin(on_disk)].reset_index(drop=True)

print(f"Unique patients : {len(patient_df):,}")
print(f"Pneumonia (1)   : {patient_df['Target'].sum():,} ({patient_df['Target'].mean()*100:.1f}%)")
print(f"No Pneumonia (0): {(patient_df['Target']==0).sum():,}")
patient_df.head()

---
## Section 3: Feature Extraction

Each DICOM image is converted to a flat feature vector:

| Component | Size | Description |
|-----------|------|-------------|
| Pixel values | 4,096 | 64×64 resized image, normalised [0,1] |
| Global stats | 5 | mean, std, p25, p50, p75 |
| **Total** | **4,101** | **per patient** |

In [ ]:
def extract_features(patient_id: str, img_dir: str = IMG_DIR) -> np.ndarray:
    """Return a (4101,) float32 feature vector for one patient.

    The vector must be identical at training and inference time.
    FEATURE_IMG_SIZE and the 5 statistics below are the shared contract.
    """
    path = os.path.join(img_dir, f"{patient_id}.dcm")
    dcm  = pydicom.dcmread(path, force=True)          # force=True: some files lack DICM header
    img  = dcm.pixel_array.astype(np.float32)         # natively grayscale (H, W)
    img  = cv2.resize(img, FEATURE_IMG_SIZE)          # bilinear → 64×64
    mx   = img.max()
    if mx > 0:
        img = img / mx                                # per-image normalise → [0, 1]
    pixels = img.flatten()                            # shape (4096,)
    stats  = np.array([
        pixels.mean(),
        pixels.std(),
        np.percentile(pixels, 25),
        np.percentile(pixels, 50),
        np.percentile(pixels, 75),
    ])                                                # shape (5,)
    return np.concatenate([pixels, stats]).astype(np.float32)

In [ ]:
# Feature extraction is slow the first time (~10–15 min for 26K patients).
# Cached to disk as features_cache.npz so subsequent runs load instantly.
if os.path.exists(FEATURE_CACHE):
    print(f"Loading cached features from {FEATURE_CACHE} ...")
    cache     = np.load(FEATURE_CACHE, allow_pickle=True)
    X_all     = cache["X"]
    y_all     = cache["y"]
    all_pids  = list(cache["pids"])
    print(f"Loaded: X_all={X_all.shape}, y_all={y_all.shape}")
else:
    print(f"Extracting features for {len(patient_df):,} patients ...")
    all_pids  = patient_df["patientId"].values
    all_labels = patient_df["Target"].values

    X_list = []
    failed = []
    for pid in tqdm(all_pids, desc="Extracting", unit="img"):
        try:
            X_list.append(extract_features(pid))
        except Exception:
            # Keep index alignment: insert a zero vector if a file is unreadable
            X_list.append(np.zeros(N_FEATURES, dtype=np.float32))
            failed.append(pid)

    X_all = np.array(X_list, dtype=np.float32)
    y_all = all_labels.astype(np.int32)

    np.savez_compressed(FEATURE_CACHE, X=X_all, y=y_all, pids=all_pids)
    print(f"Saved cache → {FEATURE_CACHE}")
    if failed:
        print(f"Failed to read {len(failed)} files (zero-padded): {failed[:5]}")

print(f"Feature matrix : {X_all.shape}  ({X_all.nbytes/1e6:.0f} MB)")
print(f"Label vector   : {y_all.shape}")

---
## Section 4: Train / Validation / Test Split

In [ ]:
# Stratified 70 / 15 / 15 split — same proportions as the CNN project for fair comparison.
# stratify=y ensures 22.5% pneumonia rate is preserved in every partition.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=42
)
# Second split: 15% of total = 15/85 of the remaining 85%
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=round(0.15 / 0.85, 4),
    stratify=y_train_full,
    random_state=42
)

print("Split summary:")
for name, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    pos = y.sum()
    print(f"  {name:5s}: {len(y):,} samples | {pos:,} pneumonia ({pos/len(y)*100:.1f}%)")

---
## Section 5: Model Training

Each candidate is a `sklearn.Pipeline` combining `StandardScaler` with a classifier.
Saving the Pipeline with joblib bundles scaler + model in one file — no separate
preprocessing step needed at inference time.

| Model | Key Hyperparameter | Class Imbalance Strategy |
|-------|-------------------|-------------------------|
| Logistic Regression | C=0.1, lbfgs | class_weight="balanced" |
| Random Forest | 300 trees, max_depth=20 | class_weight="balanced" |
| Gradient Boosting | 150 trees, lr=0.05 | subsample + scale_pos_weight |
| LinearSVC (calibrated) | C=0.1 | class_weight="balanced" |

In [ ]:
# Compute scale_pos_weight for GradientBoosting (sklearn doesn't support class_weight).
# scale_pos_weight = n_negative / n_positive mirrors the "balanced" strategy.
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos = neg_count / pos_count
print(f"scale_pos_weight for GradientBoosting: {scale_pos:.2f}")

candidates = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=0.1,                    # L2 regularisation; lower C = stronger regularisation
            class_weight="balanced",
            solver="lbfgs",
            max_iter=500,
            n_jobs=-1,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),  # RF is scale-invariant but keeps interface consistent
        ("clf", RandomForestClassifier(
            n_estimators=300,
            max_depth=20,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,            # row subsampling reduces overfitting
            random_state=42
        ))
    ]),

    "LinearSVC (calibrated)": Pipeline([
        ("scaler", StandardScaler()),
        # LinearSVC does not output probabilities — CalibratedClassifierCV adds
        # Platt scaling so we get predict_proba() for AUC computation.
        ("clf", CalibratedClassifierCV(
            LinearSVC(C=0.1, class_weight="balanced", max_iter=2000, random_state=42),
            cv=3
        ))
    ]),
}

print(f"\nDefined {len(candidates)} candidate models.")

In [ ]:
# Train each pipeline on the training set and evaluate AUC on the validation set.
# Validation AUC is used to select the best model — the test set is held out.
results = {}
for name, pipeline in candidates.items():
    print(f"Training {name} ...", end=" ", flush=True)
    pipeline.fit(X_train, y_train)
    y_val_prob = pipeline.predict_proba(X_val)[:, 1]
    val_auc    = roc_auc_score(y_val, y_val_prob)
    results[name] = {
        "pipeline":  pipeline,
        "val_auc":   val_auc,
        "val_probs": y_val_prob,
    }
    print(f"Val AUC = {val_auc:.4f}")

print("\nAll models trained.")

---
## Section 6: Model Comparison

In [ ]:
# Comparison table: sort by val_auc descending
comparison_df = pd.DataFrame([
    {"Model": name, "Val AUC": f"{r['val_auc']:.4f}"}
    for name, r in sorted(results.items(), key=lambda x: -x[1]["val_auc"])
])
print("=" * 40)
print("  MODEL COMPARISON — Validation AUC")
print("=" * 40)
print(comparison_df.to_string(index=False))
print("=" * 40)
print(f"  CNN from Scratch (baseline) : 0.7676  (from deep learning project)")

# ROC curves for all models on the validation set
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["steelblue", "tomato", "seagreen", "mediumpurple"]
for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_val, r["val_probs"])
    ax.plot(fpr, tpr, lw=2, color=color, label=f"{name} (AUC={r['val_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC=0.50)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.set_title("ROC Curves — Validation Set")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("roc_comparison.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: roc_comparison.png")

---
## Section 7: Best Model Export — joblib

The best-performing pipeline (by validation AUC) is saved as a single `.joblib` file.  
The file contains the **full sklearn Pipeline** — `StandardScaler` + classifier — so
inference requires only `joblib.load()` followed by `pipeline.predict_proba()`.
No separate preprocessing step is needed.

In [ ]:
# Select the best model by validation AUC
best_name     = max(results, key=lambda k: results[k]["val_auc"])
best_pipeline = results[best_name]["pipeline"]
best_val_auc  = results[best_name]["val_auc"]

print(f"Best model   : {best_name}")
print(f"Val AUC      : {best_val_auc:.4f}")
print(f"Pipeline     : {best_pipeline.named_steps}")

# Save the complete pipeline with joblib
joblib.dump(best_pipeline, MODEL_OUTPUT, compress=3)  # compress=3 balances size vs speed
file_kb = os.path.getsize(MODEL_OUTPUT) / 1024
print(f"\nSaved → {MODEL_OUTPUT}  ({file_kb:.0f} KB)")

# Round-trip verification: reload and confirm predictions are identical
loaded_pipeline = joblib.load(MODEL_OUTPUT)
orig_probs      = best_pipeline.predict_proba(X_val[:10])[:, 1]
loaded_probs    = loaded_pipeline.predict_proba(X_val[:10])[:, 1]
match           = np.allclose(orig_probs, loaded_probs)
print(f"Round-trip check (10 samples): {'PASS ✓' if match else 'FAIL ✗'}")

---
## Section 8: Test Set Evaluation

In [ ]:
# Evaluate on the held-out test set using the saved (reloaded) pipeline.
# Threshold = 0.35 is used for the classification report to favour recall;
# AUC is computed from raw probabilities and is threshold-independent.
DECISION_THRESHOLD = 0.35

y_test_prob = loaded_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= DECISION_THRESHOLD).astype(int)
test_auc    = roc_auc_score(y_test, y_test_prob)

print(f"Test set   : {len(y_test):,} samples  |  {y_test.sum():,} pneumonia ({y_test.mean()*100:.1f}%)")
print(f"Threshold  : {DECISION_THRESHOLD}")
print(f"Test AUC   : {test_auc:.4f}")
print()
print(classification_report(
    y_test, y_test_pred,
    target_names=["No Pneumonia (0)", "Pneumonia (1)"]
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Test Evaluation — {best_name} (joblib pipeline)", fontsize=13, fontweight="bold")

# Confusion matrix at the chosen threshold
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["No Pneumonia", "Pneumonia"],
    yticklabels=["No Pneumonia", "Pneumonia"],
    ax=axes[0]
)
axes[0].set_title(f"Confusion Matrix (threshold = {DECISION_THRESHOLD})")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# ROC curve — shows the full range of threshold operating points
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
axes[1].plot(fpr, tpr, color="steelblue", lw=2, label=f"{best_name} (AUC={test_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC=0.50)")
axes[1].set_title("ROC Curve — Test Set")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate (Recall)")
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("evaluation_plots_ml.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: evaluation_plots_ml.png")

---
## Section 9: Inference Demo

Demonstrates how any downstream system loads and uses the exported `.joblib` model.
For CLI usage: `python inference.py path/to/patient.dcm --threshold 0.35`

In [ ]:
# Minimal inference pattern — this is all that's needed in a production service.
import joblib, numpy as np, pydicom, cv2, os

FEATURE_IMG_SIZE = (64, 64)

def predict_dicom(dcm_path: str, model_path: str = "best_model_pipeline.joblib",
                  threshold: float = 0.35) -> dict:
    """Load one DICOM file and return a prediction dict using the exported pipeline."""
    pipeline = joblib.load(model_path)

    # Feature extraction — must mirror Section 3 exactly
    dcm  = pydicom.dcmread(dcm_path, force=True)
    img  = dcm.pixel_array.astype(np.float32)
    img  = cv2.resize(img, FEATURE_IMG_SIZE)
    mx   = img.max()
    if mx > 0:
        img = img / mx
    pixels = img.flatten()
    stats  = np.array([pixels.mean(), pixels.std(),
                        np.percentile(pixels, 25),
                        np.percentile(pixels, 50),
                        np.percentile(pixels, 75)])
    features = np.concatenate([pixels, stats]).reshape(1, -1)

    prob  = pipeline.predict_proba(features)[0][1]
    label = "PNEUMONIA" if prob >= threshold else "NORMAL"
    return {"file": dcm_path, "probability": round(float(prob), 4), "prediction": label}


# Demo: run inference on 5 test-set patients
import pandas as pd

demo_rows = []
for pid, true_label in zip(
    patient_df["patientId"].values[-5:],
    patient_df["Target"].values[-5:]
):
    dcm_path = os.path.join(IMG_DIR, f"{pid}.dcm")
    result = predict_dicom(dcm_path)
    result["true_label"] = int(true_label)
    result["correct"]    = (result["prediction"] == "PNEUMONIA") == bool(true_label)
    demo_rows.append(result)

demo_df = pd.DataFrame(demo_rows)[["file", "true_label", "probability", "prediction", "correct"]]
demo_df["file"] = demo_df["file"].apply(os.path.basename)  # shorten for display
print("Inference demo — 5 test patients:")
print(demo_df.to_string(index=False))

---
## Summary

| Item | Value |
|------|-------|
| Feature vector | 4,101 values (64×64 pixels + 5 stats) |
| Models compared | Logistic Regression, Random Forest, Gradient Boosting, LinearSVC |
| Selection metric | Validation ROC-AUC |
| Output file | `best_model_pipeline.joblib` |
| Inference entry point | `inference.py` |
| CNN baseline AUC (DL project) | 0.7676 |

### Deployment
```python
import joblib
pipeline = joblib.load("best_model_pipeline.joblib")
# pipeline.predict_proba(features)[:, 1]  →  P(pneumonia)
```
The loaded object is a scikit-learn `Pipeline` — call `.predict_proba()` directly.  
No GPU, no TensorFlow, no separate preprocessing step required.